In [1]:
#!pip install -q --upgrade transformers pandas numpy tqdm

In [2]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [3]:
# Input folders
SNAPSHOT_ROOT = Path("/mnt/primary/Finnhub Pipeline/finnhub_snapshots_filtered")

ANSWER_ROOT = Path("/mnt/primary/Finnhub Pipeline/finnhub_answers")

OUTPUT_ROOT = Path("/mnt/primary/Finnhub Pipeline/finbert_scores")
DAILY_JSON_ROOT = OUTPUT_ROOT / "daily_finbert_json"

SEPARATE_CSV_PATH = OUTPUT_ROOT / "finbert_news_answer_daily_scores.csv"
PROCESSING_LOG_PATH = OUTPUT_ROOT / "finbert_processing_log.csv"
COVERAGE_REPORT_PATH = OUTPUT_ROOT / "daily_coverage_finbert_report.csv"
INVALID_SCORE_FILES_PATH = OUTPUT_ROOT / "invalid_finbert_score_files.csv"

START_DATE = pd.Timestamp("2026-07-15")

MODEL_NAME = "ProsusAI/finbert"

PROCESSOR_VERSION = "finbert_separate_news_answers_v2"

MAX_CONTENT_TOKENS = 480
CHUNK_OVERLAP_TOKENS = 50
BATCH_SIZE = 16

FORCE_REPROCESS = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DAILY_JSON_ROOT.mkdir(parents=True, exist_ok=True)

print("Filtered snapshot root:", SNAPSHOT_ROOT)
print("Answer root:", ANSWER_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Separate score CSV:", SEPARATE_CSV_PATH)


Filtered snapshot root: /mnt/primary/Finnhub Pipeline/finnhub_snapshots_filtered
Answer root: /mnt/primary/Finnhub Pipeline/finnhub_answers
Output root: /mnt/primary/Finnhub Pipeline/finbert_scores_separate
Separate score CSV: /mnt/primary/Finnhub Pipeline/finbert_scores_separate/finbert_news_answer_daily_scores.csv


In [4]:
if not SNAPSHOT_ROOT.exists():
    raise FileNotFoundError(f"Snapshot folder was not found: {SNAPSHOT_ROOT.resolve()}")

if not ANSWER_ROOT.exists():
    raise FileNotFoundError(f"Answer folder was not found: {ANSWER_ROOT.resolve()}")

snapshot_date_folders = sorted(path for path in SNAPSHOT_ROOT.iterdir() if path.is_dir())

answer_date_folders = sorted(path for path in ANSWER_ROOT.iterdir() if path.is_dir())

print("Snapshot date folders:", len(snapshot_date_folders))
print("Answer date folders:", len(answer_date_folders))

print("\nLatest snapshot folders:")
for path in snapshot_date_folders[-5:]:
    print(" ", path.name)

print("\nLatest answer folders:")
for path in answer_date_folders[-5:]:
    print(" ", path.name)

Snapshot date folders: 44
Answer date folders: 48

Latest snapshot folders:
  2026-08-24
  2026-08-25
  2026-08-26
  2026-08-27
  2026-08-28

Latest answer folders:
  2026-08-27
  2026-08-28
  2026-08-29
  2026-08-30
  2026-08-31


In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Model labels:", model.config.id2label)
print("Tokenizer maximum length:", tokenizer.model_max_length)

Device: cpu
Model labels: {0: 'positive', 1: 'negative', 2: 'neutral'}
Tokenizer maximum length: 512


In [6]:
def normalise_whitespace(text: str) -> str:
    text = str(text).replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_date_from_folder(path: Path) -> pd.Timestamp | None:
    try:
        return pd.Timestamp(path.name).normalize()
    except Exception:
        return None


def safe_filename(value: str) -> str:
    value = re.sub(r'[<>:"/\\|?*]', "_", str(value))
    value = re.sub(r"\s+", " ", value).strip()
    return value


def normalise_company_key(value: str) -> str:
    value = str(value).lower()
    value = value.replace("&", "and")
    value = re.sub(r"_answer_\d{4}-\d{2}-\d{2}$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"[^a-z0-9]+", "", value)
    return value

In [7]:
def chunk_text_by_tokens(
    text: str,
    tokenizer,
    max_tokens: int = MAX_CONTENT_TOKENS,
    overlap_tokens: int = CHUNK_OVERLAP_TOKENS,
) -> list[dict[str, Any]]:
    cleaned = normalise_whitespace(text)

    if not cleaned:
        return []

    token_ids = tokenizer.encode(cleaned, add_special_tokens=False)

    if not token_ids:
        return []

    if overlap_tokens >= max_tokens:
        raise ValueError("CHUNK_OVERLAP_TOKENS must be smaller than MAX_CONTENT_TOKENS.")

    chunks = []
    start = 0
    chunk_index = 0

    while start < len(token_ids):
        end = min(start + max_tokens, len(token_ids))
        chunk_token_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        chunks.append({"chunk_index": chunk_index,"text": chunk_text,"token_count": len(chunk_token_ids)})

        if end >= len(token_ids):
            break

        start = end - overlap_tokens
        chunk_index += 1

    return chunks

In [8]:
@torch.inference_mode()
def score_text_batch(texts: list[str], batch_size: int = BATCH_SIZE) -> list[dict[str, Any]]:
    if not texts:
        return []

    id_to_label = {int(index): str(label).lower()for index, label in model.config.id2label.items()}

    all_results = []

    for batch_start in range(0, len(texts), batch_size):
        batch_texts = texts[batch_start : batch_start + batch_size]

        encoded = tokenizer(batch_texts,padding=True,truncation=True,max_length=512,return_tensors="pt")

        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

        outputs = model(**encoded)
        probabilities = torch.softmax(outputs.logits,dim=-1).cpu().numpy()

        for probability_vector in probabilities:
            probability_by_label = {
                id_to_label[index]: float(probability)
                for index, probability in enumerate(probability_vector)
            }

            positive = probability_by_label.get("positive", 0.0)
            negative = probability_by_label.get("negative", 0.0)
            neutral = probability_by_label.get("neutral", 0.0)

            sentiment = positive - negative
            confidence = max(positive, negative, neutral)
            combined = sentiment * confidence

            predicted_label = max(
                probability_by_label,
                key=probability_by_label.get
            )

            all_results.append(
                {
                    "positive_probability": positive,
                    "negative_probability": negative,
                    "neutral_probability": neutral,
                    "sentiment_score": sentiment,
                    "confidence_score": confidence,
                    "combined_score": combined,
                    "predicted_label": predicted_label
                }
            )

    return all_results

In [9]:
def aggregate_scored_chunks(scored_chunks: list[dict[str, Any]]) -> dict[str, Any] | None:
    if not scored_chunks:
        return None

    weights = np.array([max(int(chunk.get("token_count", 1)), 1) for chunk in scored_chunks],dtype=float)
    weights = weights / weights.sum()

    positive = float(np.sum([weight * chunk["positive_probability"] for weight, chunk in zip(weights, scored_chunks)]))

    negative = float(np.sum([weight * chunk["negative_probability"] for weight, chunk in zip(weights, scored_chunks)]))

    neutral = float(np.sum([weight * chunk["neutral_probability"] for weight, chunk in zip(weights, scored_chunks)]))

    sentiment = positive - negative
    confidence = max(positive, negative, neutral)
    combined = sentiment * confidence

    label_probabilities = {"positive": positive,"negative": negative,"neutral": neutral}

    return {
        "positive_probability": positive,
        "negative_probability": negative,
        "neutral_probability": neutral,
        "sentiment_score": sentiment,
        "confidence_score": confidence,
        "combined_score": combined,
        "predicted_label": max(
            label_probabilities,
            key=label_probabilities.get,
        ),
        "chunk_count": len(scored_chunks),
        "token_count": int(
            sum(
                chunk.get("token_count", 0)
                for chunk in scored_chunks
            )
        ),
    }


def score_document(text: str, source_name: str) -> dict[str, Any] | None:
    chunks = chunk_text_by_tokens(
        text=text,
        tokenizer=tokenizer,
    )

    if not chunks:
        return None

    scores = score_text_batch([chunk["text"] for chunk in chunks])

    scored_chunks = [{**chunk, **score} for chunk, score in zip(chunks, scores)]

    aggregate = aggregate_scored_chunks(scored_chunks)

    if aggregate is None:
        return None

    return {"source": source_name, **aggregate, "chunks": scored_chunks}

In [10]:
def load_snapshot_file(snapshot_path: Path) -> dict[str, Any]:
    with open(snapshot_path, "r", encoding="utf-8") as file:
        snapshot = json.load(file)

    snapshot_date = pd.to_datetime(
        snapshot.get("snapshot_date"),
        errors="coerce"
    )

    if pd.isna(snapshot_date):
        snapshot_date = parse_date_from_folder(
            snapshot_path.parent
        )

    ticker = normalise_whitespace(snapshot.get("ticker", "")).upper()

    company_name = normalise_whitespace(snapshot.get("company_name", ""))

    if snapshot_date is None or pd.isna(snapshot_date):
        raise ValueError(f"No valid snapshot date found in {snapshot_path}")

    if not ticker:
        raise ValueError(f"No ticker found inside {snapshot_path}")

    if not company_name:
        company_name = ticker

    return {
        "snapshot_date": pd.Timestamp(snapshot_date).normalize(),
        "ticker": ticker,
        "company_name": company_name,
        "category": snapshot.get("category"),
        "retrieved_at_utc": snapshot.get("retrieved_at_utc"),
        "data": snapshot.get("data", {}),
        "source_path": str(snapshot_path),
    }

In [11]:
def extract_news_items(snapshot_record: dict[str, Any]) -> list[dict[str, Any]]:
    news_items = (snapshot_record.get("data", {}).get("company_news", []))

    extracted = []
    seen_texts = set()

    for index, item in enumerate(news_items):
        if not isinstance(item, dict):
            continue

        headline = normalise_whitespace(
            item.get("headline", "")
        )
        summary = normalise_whitespace(
            item.get("summary", "")
        )

        if headline and summary:
            text = f"{headline}. {summary}"
        else:
            text = headline or summary

        if not text:
            continue

        deduplication_key = text.lower()

        if deduplication_key in seen_texts:
            continue

        seen_texts.add(deduplication_key)

        extracted.append(
            {
                "news_index": index,
                "text": text,
                "headline": headline,
                "summary": summary,
                "source": item.get("source"),
                "datetime_utc": item.get("datetime_utc"),
                "url": item.get("url"),
            }
        )

    return extracted


def score_snapshot_news(snapshot_record: dict[str, Any]) -> dict[str, Any] | None:
    news_items = extract_news_items(snapshot_record)

    if not news_items:
        return None

    scores = score_text_batch([item["text"] for item in news_items])

    scored_articles = []

    for item, score in zip(news_items, scores):
        token_count = len(
            tokenizer.encode(
                item["text"],
                add_special_tokens=False,
                truncation=True,
                max_length=MAX_CONTENT_TOKENS,
            )
        )

        scored_articles.append(
            {
                **item,
                "token_count": token_count,
                **score,
            }
        )

    aggregate = aggregate_scored_chunks(scored_articles)

    if aggregate is None:
        return None

    return {
        "source": "finnhub_news",
        **aggregate,
        "article_count": len(scored_articles),
        "articles": scored_articles,
    }

In [12]:
def build_answer_index(answer_root: Path, start_date: pd.Timestamp) -> dict[tuple[pd.Timestamp, str], Path]:
    answer_index = {}

    for date_directory in sorted(answer_root.iterdir()):
        if not date_directory.is_dir():
            continue

        folder_date = parse_date_from_folder(date_directory)

        if folder_date is None or folder_date < start_date:
            continue

        for answer_path in date_directory.glob("*.txt"):
            company_part = re.sub(r"_answer_\d{4}-\d{2}-\d{2}$", "", answer_path.stem, flags=re.IGNORECASE)
            company_key = normalise_company_key(company_part)
            answer_index[(folder_date, company_key)] = answer_path

    return answer_index


def find_answer_path(
    snapshot_date: pd.Timestamp,
    company_name: str,
    ticker: str,
    answer_index: dict[tuple[pd.Timestamp, str], Path]
) -> Path | None:
    candidate_keys = [
        normalise_company_key(company_name),
        normalise_company_key(company_name.replace(".", "")),
        normalise_company_key(ticker),
    ]

    for company_key in candidate_keys:
        path = answer_index.get((snapshot_date, company_key))

        if path is not None:
            return path

    same_date_candidates = [
        (key, path)
        for (date, key), path in answer_index.items()
        if date == snapshot_date
    ]

    target_key = normalise_company_key(company_name)

    for candidate_key, path in same_date_candidates:
        if (
            target_key in candidate_key
            or candidate_key in target_key
        ):
            return path

    return None

In [13]:
def load_and_clean_answer(answer_path: Path) -> str:
    text = answer_path.read_text(encoding="utf-8", errors="replace")

    lines = []

    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        if re.fullmatch(r"[-=_*]{3,}", line):
            continue

        lines.append(line)

    return "\n".join(lines)


def score_answer_file(answer_path: Path) -> dict[str, Any] | None:
    answer_text = load_and_clean_answer(answer_path)

    result = score_document(text=answer_text, source_name="llm_answers")

    if result is not None:
        result["answer_path"] = str(answer_path)

    return result

In [14]:
def discover_snapshot_files(snapshot_root: Path, start_date: pd.Timestamp) -> list[Path]:
    snapshot_files = []

    for date_directory in sorted(snapshot_root.iterdir()):
        if not date_directory.is_dir():
            continue

        folder_date = parse_date_from_folder(date_directory)

        if folder_date is None or folder_date < start_date:
            continue

        snapshot_files.extend(sorted(date_directory.rglob("*.json")))

    return snapshot_files

In [15]:
def calculate_file_sha256(file_path: Path | None, chunk_size: int = 1024 * 1024) -> str | None:
    if file_path is None:
        return None

    file_path = Path(file_path)

    if not file_path.exists():
        return None

    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def get_file_metadata(file_path: Path | None) -> dict[str, Any]:
    if file_path is None:
        return {
            "path": None,
            "exists": False,
            "size_bytes": None,
            "modified_at_utc": None,
            "sha256": None,
        }

    file_path = Path(file_path)

    if not file_path.exists():
        return {
            "path": str(file_path),
            "exists": False,
            "size_bytes": None,
            "modified_at_utc": None,
            "sha256": None,
        }

    stat = file_path.stat()

    return {
        "path": str(file_path),
        "exists": True,
        "size_bytes": stat.st_size,
        "modified_at_utc": datetime.fromtimestamp(
            stat.st_mtime,
            tz=timezone.utc,
        ).isoformat(),
        "sha256": calculate_file_sha256(file_path),
    }


def get_score_output_path(snapshot_date: pd.Timestamp, company_name: str, ticker: str) -> Path:
    date_directory = (DAILY_JSON_ROOT / str(pd.Timestamp(snapshot_date).date()))

    date_directory.mkdir(parents=True,exist_ok=True)

    return (date_directory/ (f"{safe_filename(company_name)}_"f"{safe_filename(ticker)}_finbert.json"))


def load_existing_score(score_path: Path) -> dict[str, Any] | None:
    if not score_path.exists():
        return None
    try:
        with open(score_path, "r", encoding="utf-8") as file:
            result = json.load(file)
        required_fields = {
            "date",
            "ticker",
            "company_name",
            "model",
            "processor_version",
            "input_files",
        }
        if not required_fields.issubset(result):
            return None
        return result
    except (json.JSONDecodeError, OSError, TypeError):
        return None


def score_needs_processing(
    score_path: Path,
    snapshot_metadata: dict[str, Any],
    answer_metadata: dict[str, Any],
    force: bool = False,
) -> tuple[bool, str]:
    if force:
        return True, "forced_reprocessing"

    existing = load_existing_score(score_path)

    if existing is None:
        if score_path.exists():
            return True, "invalid_existing_score"

        return True, "score_missing"

    if existing.get("model") != MODEL_NAME:
        return True, "model_changed"

    if existing.get("processor_version") != PROCESSOR_VERSION:
        return True, "processor_version_changed"

    previous_inputs = existing.get("input_files", {})
    previous_snapshot = previous_inputs.get("snapshot", {})
    previous_answer = previous_inputs.get("answer", {})

    if (
        previous_snapshot.get("sha256")
        != snapshot_metadata.get("sha256")
    ):
        return True, "snapshot_changed"

    if (
        previous_answer.get("sha256")
        != answer_metadata.get("sha256")
    ):
        return True, "answer_added_or_changed"

    has_news_score = existing.get("news_score") is not None
    has_answer_score = existing.get("answer_score") is not None

    if not has_news_score and not has_answer_score:
        return True, "score_contains_no_sources"

    if answer_metadata.get("exists") and not has_answer_score:
        return True, "answer_score_missing"

    return False, "already_complete"


def write_json_atomically(output_path: Path, payload: dict[str, Any]) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")

    with open(temporary_path, "w", encoding="utf-8") as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )

    os.replace(temporary_path, output_path)

In [16]:
def process_one_snapshot_incrementally(
    snapshot_path: Path,
    answer_index: dict[tuple[pd.Timestamp, str], Path],
    force: bool = False,
) -> dict[str, Any]:
    snapshot = load_snapshot_file(snapshot_path)

    snapshot_date = snapshot["snapshot_date"]
    ticker = snapshot["ticker"]
    company_name = snapshot["company_name"]

    answer_path = find_answer_path(
        snapshot_date=snapshot_date,
        company_name=company_name,
        ticker=ticker,
        answer_index=answer_index,
    )

    score_path = get_score_output_path(
        snapshot_date=snapshot_date,
        company_name=company_name,
        ticker=ticker,
    )

    snapshot_metadata = get_file_metadata(snapshot_path)
    answer_metadata = get_file_metadata(answer_path)

    needs_processing, reason = score_needs_processing(
        score_path=score_path,
        snapshot_metadata=snapshot_metadata,
        answer_metadata=answer_metadata,
        force=force,
    )

    if not needs_processing:
        return {
            "Date": snapshot_date,
            "Symbol": ticker,
            "Company": company_name,
            "Snapshot_Path": str(snapshot_path),
            "Answer_Path": (
                str(answer_path)
                if answer_path is not None
                else None
            ),
            "Score_Path": str(score_path),
            "Status": "skipped",
            "Reason": reason,
            "News_Found": None,
            "Answer_Found": answer_path is not None,
            "Error": None,
        }

    # --------------------------------------------------------
    # 1. NEWS FEATURE
    # --------------------------------------------------------
    news_score = score_snapshot_news(snapshot)

    # --------------------------------------------------------
    # 2. LLM-ANSWER FEATURE
    # --------------------------------------------------------
    answer_score = None

    if answer_path is not None:
        answer_score = score_answer_file(answer_path)

    if news_score is None and answer_score is None:
        raise ValueError(
            "Neither filtered snapshot news nor answer text produced a score "
            f"for {ticker} on {snapshot_date.date()}."
        )

    company_result = {
        "date": str(snapshot_date.date()),
        "ticker": ticker,
        "company_name": company_name,
        "category": snapshot.get("category"),
        "model": MODEL_NAME,
        "processor_version": PROCESSOR_VERSION,
        "processed_at_utc": datetime.now(timezone.utc).isoformat(),
        "processing_reason": reason,
        "scoring_definition": {
            "sentiment_score": (
                "positive_probability - negative_probability"
            ),
            "confidence_score": "maximum class probability",
            "combined_score_per_source": (
                "sentiment_score * confidence_score"
            ),
            "source_fusion": (
                "None. News and LLM-answer scores are stored separately."
            ),
        },
        "input_files": {
            "snapshot": snapshot_metadata,
            "answer": answer_metadata,
        },

        # Independent source outputs
        "news_score": news_score,
        "answer_score": answer_score,

        "snapshot_path": str(snapshot_path),
        "answer_path": (
            str(answer_path)
            if answer_path is not None
            else None
        ),
    }

    write_json_atomically(
        output_path=score_path,
        payload=company_result,
    )

    return {
        "Date": snapshot_date,
        "Symbol": ticker,
        "Company": company_name,
        "Snapshot_Path": str(snapshot_path),
        "Answer_Path": (
            str(answer_path)
            if answer_path is not None
            else None
        ),
        "Score_Path": str(score_path),
        "Status": "processed",
        "Reason": reason,
        "News_Found": news_score is not None,
        "Answer_Found": answer_score is not None,
        "Error": None,
    }


In [17]:
def flatten_saved_score(result: dict[str, Any], score_path: Path) -> dict[str, Any]:
    news = result.get("news_score") or {}
    answer = result.get("answer_score") or {}
    input_files = result.get("input_files", {})

    return {
        "Date": pd.to_datetime(
            result.get("date"),
            errors="coerce",
        ),
        "Symbol": result.get("ticker"),
        "Company": result.get("company_name"),
        "Category": result.get("category"),

        "News_Positive_Probability": news.get(
            "positive_probability"
        ),
        "News_Negative_Probability": news.get(
            "negative_probability"
        ),
        "News_Neutral_Probability": news.get(
            "neutral_probability"
        ),
        "News_Sentiment": news.get(
            "sentiment_score"
        ),
        "News_Confidence": news.get(
            "confidence_score"
        ),
        "News_Combined": news.get(
            "combined_score"
        ),
        "News_Article_Count": news.get(
            "article_count"
        ),


        "Answer_Positive_Probability": answer.get(
            "positive_probability"
        ),
        "Answer_Negative_Probability": answer.get(
            "negative_probability"
        ),
        "Answer_Neutral_Probability": answer.get(
            "neutral_probability"
        ),
        "Answer_Sentiment": answer.get(
            "sentiment_score"
        ),
        "Answer_Confidence": answer.get(
            "confidence_score"
        ),
        "Answer_Combined": answer.get(
            "combined_score"
        ),
        "Answer_Chunk_Count": answer.get(
            "chunk_count"
        ),

        "Model": result.get("model"),
        "Processor_Version": result.get(
            "processor_version"
        ),
        "Processed_At_UTC": result.get(
            "processed_at_utc"
        ),

        "Snapshot_SHA256": (
            input_files.get(
                "snapshot",
                {},
            ).get("sha256")
        ),
        "Answer_SHA256": (
            input_files.get(
                "answer",
                {},
            ).get("sha256")
        ),

        "Snapshot_Path": result.get(
            "snapshot_path"
        ),
        "Answer_Path": result.get(
            "answer_path"
        ),
        "Score_Path": str(score_path),
    }


def rebuild_separate_scores_csv() -> pd.DataFrame:
    rows = []
    invalid_files = []

    score_files = sorted(
        DAILY_JSON_ROOT.rglob(
            "*_finbert.json"
        )
    )

    for score_path in score_files:
        try:
            with open(
                score_path,
                "r",
                encoding="utf-8",
            ) as file:
                result = json.load(file)

            row = flatten_saved_score(
                result=result,
                score_path=score_path,
            )

            if (
                pd.isna(row["Date"])
                or
                not row["Symbol"]
            ):
                raise ValueError(
                    "Missing Date or Symbol."
                )

            rows.append(row)

        except Exception as error:
            invalid_files.append(
                {
                    "Score_Path": str(
                        score_path
                    ),
                    "Error": (
                        f"{type(error).__name__}: "
                        f"{error}"
                    ),
                }
            )

    scores_df = pd.DataFrame(rows)

    if not scores_df.empty:
        scores_df["Date"] = pd.to_datetime(
            scores_df["Date"]
        ).dt.normalize()

        scores_df["Symbol"] = (
            scores_df["Symbol"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        scores_df = (
            scores_df
            .sort_values(
                [
                    "Date",
                    "Symbol",
                    "Processed_At_UTC",
                ]
            )
            .drop_duplicates(
                subset=[
                    "Date",
                    "Symbol",
                ],
                keep="last",
            )
            .reset_index(drop=True)
        )

        scores_df.to_csv(
            SEPARATE_CSV_PATH,
            index=False,
        )

    invalid_df = pd.DataFrame(
        invalid_files
    )

    invalid_df.to_csv(
        INVALID_SCORE_FILES_PATH,
        index=False,
    )

    return scores_df


In [18]:
def run_incremental_finbert_update(force: bool = FORCE_REPROCESS) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("Building answer-file index...")

    current_answer_index = build_answer_index(
        answer_root=ANSWER_ROOT,
        start_date=START_DATE,
    )

    print(
        "Answer files indexed:",
        len(current_answer_index),
    )

    print(
        "\nDiscovering FILTERED snapshot files..."
    )

    current_snapshot_files = discover_snapshot_files(
        snapshot_root=SNAPSHOT_ROOT,
        start_date=START_DATE,
    )

    print(
        "Filtered snapshot files discovered:",
        len(current_snapshot_files),
    )

    processing_records = []

    for snapshot_path in tqdm(
        current_snapshot_files,
        desc="Checking separate FinnBERT scores",
    ):
        try:
            record = process_one_snapshot_incrementally(
                snapshot_path=snapshot_path,
                answer_index=current_answer_index,
                force=force,
            )

            processing_records.append(
                record
            )

        except Exception as error:
            processing_records.append(
                {
                    "Date": None,
                    "Symbol": None,
                    "Company": None,
                    "Snapshot_Path": str(
                        snapshot_path
                    ),
                    "Answer_Path": None,
                    "Score_Path": None,
                    "Status": "failed",
                    "Reason": "processing_error",
                    "News_Found": None,
                    "Answer_Found": None,
                    "Error": (
                        f"{type(error).__name__}: "
                        f"{error}"
                    ),
                }
            )

    run_log = pd.DataFrame(
        processing_records
    )

    run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_log_path = (OUTPUT_ROOT/ f"processing_log_{run_timestamp}.csv")

    run_log.to_csv(run_log_path, index=False)

    run_log.to_csv(
        PROCESSING_LOG_PATH,
        index=False,
    )

    print("\nRebuilding separate News / Answer score CSV...")

    scores_df = rebuild_separate_scores_csv()

    print("\nRun status:")

    if not run_log.empty:
        print(run_log["Status"].value_counts(dropna=False).to_string())

    if (not run_log.empty and "Reason" in run_log.columns):
        print("\nProcessing reasons:")
        print(run_log["Reason"].value_counts(dropna=False).to_string())

    if not scores_df.empty:
        print("\nSeparate score dataset")
        print("Rows:", len(scores_df))
        print("Companies:", scores_df["Symbol"].nunique())
        print("First date:", scores_df["Date"].min().date())
        print("Last date:", scores_df["Date"].max().date())
        print("Saved to:", SEPARATE_CSV_PATH)

    return scores_df, run_log


In [19]:
scores_df, run_log = run_incremental_finbert_update()

Building answer-file index...
Answer files indexed: 4700

Discovering FILTERED snapshot files...
Filtered snapshot files discovered: 4401


Checking separate FinnBERT scores:   0%|          | 0/4401 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (10164 > 512). Running this sequence through the model will result in indexing errors



Rebuilding separate News / Answer score CSV...

Run status:
Status
skipped      3367
processed    1034

Processing reasons:
Reason
already_complete    3367
score_missing       1000
snapshot_changed      34

Separate score dataset
Rows: 4400
Companies: 100
First date: 2026-07-15
Last date: 2026-08-28
Saved to: /mnt/primary/Finnhub Pipeline/finbert_scores_separate/finbert_news_answer_daily_scores.csv


In [20]:
def create_daily_coverage_report() -> pd.DataFrame:
    snapshot_rows = []

    for snapshot_path in discover_snapshot_files(snapshot_root=SNAPSHOT_ROOT, start_date=START_DATE):
        try:
            snapshot = load_snapshot_file(snapshot_path)
            snapshot_rows.append({"Date": snapshot["snapshot_date"],"Symbol": snapshot["ticker"]})
        except Exception:
            continue

    snapshots_df = pd.DataFrame(snapshot_rows)
    scores_df = rebuild_separate_scores_csv()

    if snapshots_df.empty:
        raise ValueError("No valid snapshots were found.")

    snapshot_coverage = (
        snapshots_df
        .drop_duplicates(["Date", "Symbol"])
        .groupby("Date")
        .agg(
            Snapshot_Companies=("Symbol", "nunique")
        )
        .reset_index()
    )

    if scores_df.empty:
        score_coverage = pd.DataFrame(
            columns=[
                "Date",
                "Scored_Companies",
                "Scores_With_Answers",
                "Scores_With_News",
            ]
        )
    else:
        score_coverage = (
            scores_df
            .groupby("Date")
            .agg(
                Scored_Companies=("Symbol", "nunique"),
                Scores_With_Answers=(
                    "Answer_Combined",
                    lambda values: int(
                        values.notna().sum()
                    ),
                ),
                Scores_With_News=(
                    "News_Combined",
                    lambda values: int(
                        values.notna().sum()
                    ),
                ),
            )
            .reset_index()
        )

    coverage = snapshot_coverage.merge(score_coverage, on="Date", how="left")

    count_columns = ["Scored_Companies", "Scores_With_Answers", "Scores_With_News"]

    for column in count_columns:
        coverage[column] = (coverage[column].fillna(0).astype(int))

    coverage["Missing_Scores"] = (coverage["Snapshot_Companies"] - coverage["Scored_Companies"])

    coverage["Complete"] = (coverage["Missing_Scores"] == 0)

    coverage = coverage.sort_values("Date").reset_index(drop=True)

    coverage.to_csv(COVERAGE_REPORT_PATH, index=False)

    return coverage


coverage_report = create_daily_coverage_report()
display(coverage_report)

,Date,Snapshot_Companies,Scored_Companies,Scores_With_Answers,Scores_With_News,Missing_Scores,Complete
0,2026-07-15,100,100,100,96,0,True
1,2026-07-16,100,100,100,97,0,True
2,2026-07-17,100,100,100,97,0,True
3,2026-07-18,100,100,100,96,0,True
4,2026-07-19,100,100,100,96,0,True
5,2026-07-20,100,100,100,96,0,True
6,2026-07-21,100,100,100,95,0,True
7,2026-07-22,100,100,100,97,0,True
8,2026-07-23,100,100,100,97,0,True
9,2026-07-24,100,100,100,96,0,True


In [21]:
failed_records = run_log[run_log["Status"] == "failed"].copy()

print("Failed records:", len(failed_records))

if not failed_records.empty:
    display(failed_records[["Snapshot_Path", "Reason", "Error"]])

Failed records: 0


In [22]:
missing_answer_scores = scores_df[scores_df["Answer_Combined"].isna()].copy()

print("Company/date scores without answer sentiment:", len(missing_answer_scores))

if not missing_answer_scores.empty:
    display(
        missing_answer_scores[
            [
                "Date",
                "Symbol",
                "Company",
                "News_Combined",
                "Answer_Path",
            ]
        ].head(50)
    )

Company/date scores without answer sentiment: 0


In [23]:
score_columns = [
    "News_Sentiment",
    "News_Confidence",
    "News_Combined",
    "Answer_Sentiment",
    "Answer_Confidence",
    "Answer_Combined"
]

display(scores_df[score_columns].describe().T)


,count,mean,std,min,25%,50%,75%,max
News_Sentiment,4145.0,0.206305,0.400377,-0.964786,-0.027190,0.242383,0.499771,0.941218
News_Confidence,4145.0,0.594760,0.152488,0.336666,0.478387,0.559065,0.692692,0.972929
News_Combined,4145.0,0.136393,0.303865,-0.938479,-0.012870,0.123912,0.293189,0.900099
Answer_Sentiment,4400.0,0.104117,0.198906,-0.709820,-0.010157,0.114054,0.241826,0.711037
Answer_Confidence,4400.0,0.464743,0.076173,0.335291,0.408482,0.451489,0.506642,0.829573
Answer_Combined,4400.0,0.052477,0.110979,-0.588847,-0.004123,0.049057,0.113728,0.577167


In [24]:
for column in [
    "News_Sentiment",
    "News_Combined",
    "Answer_Sentiment",
    "Answer_Combined"
]:
    valid = (
        scores_df[column]
        .dropna()
        .between(-1, 1)
        .all()
    )

    print(f"{column} within [-1, 1]:", valid)

for column in ["News_Confidence", "Answer_Confidence"]:
    valid = (scores_df[column].dropna().between(0, 1).all())

    print(f"{column} within [0, 1]:", valid)


News_Sentiment within [-1, 1]: True
News_Combined within [-1, 1]: True
Answer_Sentiment within [-1, 1]: True
Answer_Combined within [-1, 1]: True
News_Confidence within [0, 1]: True
Answer_Confidence within [0, 1]: True


In [25]:
TICKER_TO_INSPECT = "AAPL"

company_scores = scores_df[scores_df["Symbol"] == TICKER_TO_INSPECT].copy()

display(
    company_scores[
        [
            "Date",
            "Symbol",

            "News_Sentiment",
            "News_Confidence",
            "News_Combined",
            "News_Article_Count",

            "Answer_Sentiment",
            "Answer_Confidence",
            "Answer_Combined",
            "Answer_Chunk_Count"
        ]
    ]
)


,Date,Symbol,News_Sentiment,News_Confidence,News_Combined,News_Article_Count,Answer_Sentiment,Answer_Confidence,Answer_Combined,Answer_Chunk_Count
0,2026-07-15,AAPL,-0.168905,0.410870,-0.069398,12.0,0.364574,0.597940,0.217993,19
100,2026-07-16,AAPL,0.293684,0.534002,0.156828,12.0,0.537717,0.692466,0.372351,19
200,2026-07-17,AAPL,0.307042,0.585160,0.179669,8.0,0.225512,0.483661,0.109071,22
300,2026-07-18,AAPL,0.139637,0.512013,0.071496,12.0,0.454516,0.611199,0.277800,19
400,2026-07-19,AAPL,0.071488,0.396617,0.028354,8.0,0.479676,0.630818,0.302588,22
500,2026-07-20,AAPL,0.324663,0.463548,0.150497,10.0,0.262859,0.495368,0.130212,33
600,2026-07-21,AAPL,-0.151329,0.432527,-0.065454,11.0,0.271557,0.526716,0.143033,33
700,2026-07-22,AAPL,0.227279,0.466552,0.106038,6.0,0.436428,0.650259,0.283791,22
800,2026-07-23,AAPL,-0.294701,0.466973,-0.137617,8.0,0.205962,0.458368,0.094407,34
900,2026-07-24,AAPL,0.080396,0.386752,0.031093,5.0,0.112462,0.417202,0.046919,33


In [26]:
news_features = [
    "News_Sentiment",
    "News_Confidence",
    "News_Combined",
    "News_Article_Count"
]

answer_features = [
    "Answer_Sentiment",
    "Answer_Confidence",
    "Answer_Combined",
    "Answer_Chunk_Count"
]

compact_separate_features = [
    "News_Combined",
    "Answer_Combined"
]

print("News features:")
print(news_features)

print("\nLLM-answer features:")
print(answer_features)

print("\nRecommended compact calibration features:")
print(compact_separate_features)


News features:
['News_Sentiment', 'News_Confidence', 'News_Combined', 'News_Article_Count']

LLM-answer features:
['Answer_Sentiment', 'Answer_Confidence', 'Answer_Combined', 'Answer_Chunk_Count']

Recommended compact calibration features:
['News_Combined', 'Answer_Combined']


In [27]:
scores_df, run_log = run_incremental_finbert_update()
coverage_report = create_daily_coverage_report()

Building answer-file index...
Answer files indexed: 4700

Discovering FILTERED snapshot files...
Filtered snapshot files discovered: 4401


Checking separate FinnBERT scores:   0%|          | 0/4401 [00:00<?, ?it/s]


Rebuilding separate News / Answer score CSV...

Run status:
Status
skipped    4401

Processing reasons:
Reason
already_complete    4401

Separate score dataset
Rows: 4400
Companies: 100
First date: 2026-07-15
Last date: 2026-08-28
Saved to: /mnt/primary/Finnhub Pipeline/finbert_scores_separate/finbert_news_answer_daily_scores.csv
